In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

In [ ]:
from models.cpmp_transformer_v9 import CPMPTransformer
from training.training import load_model

model_ale     = load_model(CPMPTransformer, "v9_Original")
model_robusto = load_model(CPMPTransformer, "v9_ROBUSTO")

H_MODEL = model_ale.H  # 12 — H_pad con el que fueron entrenados ambos modelos
print(f"H_MODEL: {H_MODEL}")
print(f"v9_ALE     cargado — H={model_ale.H}")
print(f"v9_ROBUSTO cargado — H={model_robusto.H}")

AttributeError: 'CPMPTransformer' object has no attribute 'H'

In [ ]:
from solvers.model import ModelSolver

solver_ale     = ModelSolver(model_ale)
solver_robusto = ModelSolver(model_robusto)

In [ ]:
from pathlib import Path
from settings import INSTANCE_FOLDER

CVS_PATH = INSTANCE_FOLDER / "benchmarks" / "CVS"

cvs_folders = sorted(
    [d for d in os.listdir(CVS_PATH) if (CVS_PATH / d).is_dir()]
)

results = {}          # guarda listas completas para el resumen final
total_ale = total_rob = total_inst = 0

HDR = f"{'Categoría':>10}  {'H':>3} {'S':>3} {'steps':>6}  |  {'ALE':>14}  {'ROBUSTO':>14}"
SEP = "-" * len(HDR)
print(HDR)
print(SEP)

for folder_name in cvs_folders:
    H_real, S_real = [int(x) for x in folder_name.split("-")]
    max_steps = S_real * H_real * 4
    folder_rel = f"benchmarks/CVS/{folder_name}"

    solved_ale, steps_ale = solver_ale.solve_from_folder(folder_rel, H_MODEL, max_steps)
    solved_rob, steps_rob = solver_robusto.solve_from_folder(folder_rel, H_MODEL, max_steps)

    n     = len(solved_ale)
    n_ale = sum(solved_ale)
    n_rob = sum(solved_rob)

    results[folder_name] = {
        "H": H_real, "S": S_real, "n": n,
        "solved_ale":  solved_ale,  "steps_ale":  steps_ale,
        "solved_rob":  solved_rob,  "steps_rob":  steps_rob,
    }

    total_ale  += n_ale
    total_rob  += n_rob
    total_inst += n

    print(
        f"{folder_name:>10}  {H_real:>3} {S_real:>3} {max_steps:>6}  |  "
        f"{n_ale:>3}/{n:<3} ({n_ale/n:>5.1%})  "
        f"{n_rob:>3}/{n:<3} ({n_rob/n:>5.1%})"
    )

print(SEP)
print(
    f"{'TOTAL':>10}  {'':>3} {'':>3} {'':>6}  |  "
    f"{total_ale:>3}/{total_inst:<3} ({total_ale/total_inst:>5.1%})  "
    f"{total_rob:>3}/{total_inst:<3} ({total_rob/total_inst:>5.1%})"
)

In [ ]:
from solvers.utils import summary

all_solved_ale = [s for r in results.values() for s in r['solved_ale']]
all_steps_ale  = [s for r in results.values() for s in r['steps_ale']]
all_solved_rob = [s for r in results.values() for s in r['solved_rob']]
all_steps_rob  = [s for r in results.values() for s in r['steps_rob']]

print("v9_ALE:")
summary(all_solved_ale, all_steps_ale)

print("\nv9_ROBUSTO:")
summary(all_solved_rob, all_steps_rob)